# EJERCICIO 4: Análisis de subjetividad de los comentarios

In [ ]:
import json
from google.colab import drive
drive.mount('/content/drive')


with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_wallstreet_corregido", "r", encoding="utf-8") as f:
    result_walstreet_corregido = json.load(f)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_soccer_corregido", "r", encoding="utf-8") as f:
    result_soccer_corregido = json.load(f)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_nba_corregido", "r", encoding="utf-8") as f:
    result_nba_corregido = json.load(f)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_unpopular_corregido", "r", encoding="utf-8") as f:
    result_unpopular_corregido = json.load(f)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_fortnite_corregido", "r", encoding="utf-8") as f:
    result_fortnite_corregido = json.load(f)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_elden_corregido", "r", encoding="utf-8") as f:
    result_elden_corregido = json.load(f)

Mounted at /content/drive


In [ ]:
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
import numpy as np
from scipy.special import softmax
import csv
import urllib.request
task='sentiment'
MODEL = f"cardiffnlp/twitter-roberta-base-{task}"

tokenizer = AutoTokenizer.from_pretrained(MODEL)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
model.save_pretrained(MODEL)



pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# download label mapping
labels=[]
mapping_link = f"https://raw.githubusercontent.com/cardiffnlp/tweeteval/main/datasets/{task}/mapping.txt"
with urllib.request.urlopen(mapping_link) as f:
    html = f.read().decode('utf-8').split("\n")
    csvreader = csv.reader(html, delimiter='\t')
labels = [row[1] for row in csvreader if len(row) > 1]

In [ ]:
from transformers import pipeline
subreddit_list = [result_elden_corregido, result_fortnite_corregido, result_walstreet_corregido, result_unpopular_corregido, result_nba_corregido, result_soccer_corregido]
# 1. Mapeo para traducir las etiquetas del modelo
mapeo = {
    "LABEL_0": "negativo",
    "LABEL_1": "neutral",
    "LABEL_2": "positivo"
}

# 2. Crear el pipeline (añade device=0 si estás usando una GPU)
clasificador = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, truncation=True, max_length=512, top_k=None)

# 3. Recorreremos los datos
for subreddit in subreddit_list:
    for submission in subreddit.get("submissions", []):

        # Extraer solo los textos
        textos = [comment["body_preprocessed"] for comment in submission.get("comments", [])]

        if not textos:
            continue

        # Procesar todos los comentarios de golpe (batching)
        resultados = clasificador(textos)

        # 4. Asignar los resultados traduciendo y convirtiendo a float
        for comment, resultado in zip(submission.get("comments", []), resultados):

            etiqueta_cruda = resultado[0]['label']
            mejor_etiqueta = mapeo.get(etiqueta_cruda, etiqueta_cruda)
            mejor_score = float(resultado[0]['score'])

            # Guardamos la tupla como pediste (en JSON se convertirá en una lista)
            comment["sentiment"] = (mejor_etiqueta, mejor_score)

            # Guardar el resto de puntuaciones
            for item in resultado[1:]:
                etiq_traducida = mapeo.get(item['label'], item['label'])
                comment[etiq_traducida] = float(item['score'])



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
result_elden_corregido_rob_sent = result_elden_corregido
result_fortnite_corregido_rob_sent = result_fortnite_corregido
result_walstreet_corregido_rob_sent = result_walstreet_corregido
result_unpopular_corregido_rob_sent = result_unpopular_corregido
result_nba_corregido_rob_sent = result_nba_corregido
result_soccer_corregido_rob_sent = result_soccer_corregido

In [ ]:

import json

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_wallstreet_rob_sent", "w", encoding="utf-8") as f:
    json.dump(result_walstreet_corregido_rob_sent, f, ensure_ascii=False, indent=2)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_elden_rob_sent", "w", encoding="utf-8") as f:
    json.dump(result_elden_corregido_rob_sent, f, ensure_ascii=False, indent=2)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_fortnite_rob_sent", "w", encoding="utf-8") as f:
    json.dump(result_fortnite_corregido_rob_sent, f, ensure_ascii=False, indent=2)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_unpopular_rob_sent", "w", encoding="utf-8") as f:
    json.dump(result_unpopular_corregido_rob_sent, f, ensure_ascii=False, indent=2)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_nba_rob_sent", "w", encoding="utf-8") as f:
    json.dump(result_nba_corregido_rob_sent, f, ensure_ascii=False, indent=2)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_soccer_rob_sent", "w", encoding="utf-8") as f:
    json.dump(result_soccer_corregido_rob_sent, f, ensure_ascii=False, indent=2)

## Michellejieli

In [ ]:
import json
with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_wallstreet_corregido", "r", encoding="utf-8") as f:
    result_walstreet_corregido = json.load(f)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_soccer_corregido", "r", encoding="utf-8") as f:
    result_soccer_corregido = json.load(f)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_nba_corregido", "r", encoding="utf-8") as f:
    result_nba_corregido = json.load(f)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_unpopular_corregido", "r", encoding="utf-8") as f:
    result_unpopular_corregido = json.load(f)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_fortnite_corregido", "r", encoding="utf-8") as f:
    result_fortnite_corregido = json.load(f)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_elden_corregido", "r", encoding="utf-8") as f:
    result_elden_corregido = json.load(f)

In [ ]:
from transformers import pipeline
#Creamos el pipeline del modelo
clasificador = pipeline("sentiment-analysis", model="michellejieli/emotion_text_classifier", truncation=True, max_length=512)
subreddit_list = [result_elden_corregido, result_fortnite_corregido, result_walstreet_corregido, result_unpopular_corregido, result_nba_corregido, result_soccer_corregido]
for subreddit in subreddit_list:
    for submission in subreddit.get("submissions", []):

        textos = [comment["body_preprocessed"] for comment in submission.get("comments", [])]#Recogemos el cuerpo de todos los comentarios

        if not textos:
            continue

        resultados = clasificador(textos) #Clasificamos el cuerpo del comentario

        for comment, resultado in zip(submission.get("comments", []), resultados):
            etiqueta = resultado['label']
            score = float(resultado['score'])

            comment["sentiment"] = (etiqueta, score) #Guardamos la clasificación de los sentimientos del texto con su score particular

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: michellejieli/emotion_text_classifier
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
result_elden_corregido_mic_sent = result_elden_corregido
result_fortnite_corregido_mic_sent = result_fortnite_corregido
result_walstreet_corregido_mic_sent = result_walstreet_corregido
result_unpopular_corregido_mic_sent = result_unpopular_corregido
result_nba_corregido_mic_sent = result_nba_corregido
result_soccer_corregido_mic_sent = result_soccer_corregido

In [ ]:
with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_wallstreet_mic_sent", "w", encoding="utf-8") as f:
    json.dump(result_walstreet_corregido_mic_sent, f, ensure_ascii=False, indent=2)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_elden_mic_sent", "w", encoding="utf-8") as f:
    json.dump(result_elden_corregido_mic_sent, f, ensure_ascii=False, indent=2)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_fortnite_mic_sent", "w", encoding="utf-8") as f:
    json.dump(result_fortnite_corregido_mic_sent, f, ensure_ascii=False, indent=2)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_unpopular_mic_sent", "w", encoding="utf-8") as f:
    json.dump(result_unpopular_corregido_mic_sent, f, ensure_ascii=False, indent=2)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_nba_mic_sent", "w", encoding="utf-8") as f:
    json.dump(result_nba_corregido_mic_sent, f, ensure_ascii=False, indent=2)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_soccer_mic_sent", "w", encoding="utf-8") as f:
    json.dump(result_soccer_corregido_mic_sent, f, ensure_ascii=False, indent=2)